# Quantum Approximate Optimization Algorithm (QAOA)
In materials science research, Quantum Annealing and its gate-based sibling, the Quantum Approximate Optimization Algorithm (QAOA), are heavily used to discover optimal molecular structures or material properties.

A major research challenge in metallurgical engineering and advanced manufacturing is Crystal Grain Boundary Optimization. When a metal or ceramic is physically annealed (heated and slowly cooled), small crystalline regions called "grains" form. The boundaries where these micro-crystals meet are structurally weak; a major goal of materials science is to discover a crystal layout configuration that minimizes overall surface energy to prevent structural fractures.

We can formulate this materials optimization problem using a hybrid QAOA architecture. We will map a crystal lattice network to a quantum cost Hamiltonian, simulate the material's structural stress, and train a quantum circuit to find the globally stable "lowest energy" grain configuration.The Materials Problem: Lattice Spin-Glass MinimizationWe model a piece of material as a grid of localized crystal domains. Each domain can adopt one of two magnetic or structural orientations (represented as a binary spin state, $+1$ or $-1$).The total energy of the material interface is determined by the interactions between neighboring grains:$$H_{\text{Material}} = \sum_{\langle i,j \rangle} J_{ij} Z_i Z_j + \sum_i h_i Z_i$$Where $J_{ij}$ represents the structural tension between adjacent grains $i$ and $j$, and $h_i$ represents localized internal material stress. The goal is to discover the configuration of $Z$ states that minimizes the energy.

In [ ]:
# Install required packages (quiet mode for cleaner output)
!pip install --quiet pennylane
!pip install --quiet amazon-braket-pennylane-plugin

### Step 1: Defining the Crystal Structural Map

In [ ]:
import pennylane as qml
from pennylane import numpy as np

# Let's model a 4-grain linear junction on a material surface
num_grains = 4

# Define structural tension (J) between neighboring crystal boundaries
# Negative values mean the grains prefer to align; positive means they push apart
structural_tension = {
    (0, 1): 1.0,   # High stress interface
    (1, 2): -0.5,  # Stable interface
    (2, 3): 1.2,   # Extreme stress interface
    (0, 3): 0.3    # Boundary overlap
}

# Localized internal material defects/stress (h) forcing specific grains to warp
internal_stress = [0.2, -0.1, 0.4, 0.0]

# Construct the Material Cost Hamiltonian matrix using PauliZ operators
obs = []
coeffs = []

for (i, j), J in structural_tension.items():
    coeffs.append(J)
    obs.append(qml.PauliZ(i) @ qml.PauliZ(j))

for i, h in enumerate(internal_stress):
    if h != 0.0:
        coeffs.append(h)
        obs.append(qml.PauliZ(i))

H_cost = qml.Hamiltonian(coeffs, obs)

#Step 2: Designing the QAOA "Quantum Cooling" Ansatz
A QAOA circuit simulates physical annealing by alternating between two layers:
1.   The Cost Layer ($H_{\text{cost}}$): Evolves the qubits based on the structural stress of the material.
2.   The Mixer Layer ($H_{\text{mixer}}$): Applies global $X$ rotations to induce quantum tunneling, allowing the system to escape unfavorable local minima.

In [ ]:
dev = qml.device("braket.local.qubit", wires=num_grains)

# The Mixer Hamiltonian handles quantum fluctuations
def mixer_layer(beta):
    for i in range(num_grains):
        qml.RX(2 * beta, wires=i)

# The Cost Layer maps structural stress to phase shifts
def cost_layer(gamma):
    qml.CommutingEvolution(H_cost, 2 * gamma)

@qml.qnode(dev)
def qaoa_annealer(gamma_params, beta_params):
    # Step 1: Generate a uniform initial superposition (unstructured material state)
    for i in range(num_grains):
        qml.Hadamard(wires=i)

    # Step 2: Apply p layers of alternating cost and mixer evolutions
    # Higher layers simulate a slower, more accurate cooling schedule
    p = len(gamma_params)
    for layer in range(p):
        cost_layer(gamma_params[layer])
        mixer_layer(beta_params[layer])

    # Sample the final configuration probabilities of the crystal grains
    return qml.probs(wires=range(num_grains))

### Step 3: Training the Hybrid Annealing Solver
We will optimize the parameters $\gamma$ and $\beta$. In materials research, this classical-quantum loop is viewed as tuning the automated cooling rate profile of the system.

In [ ]:
# Initialize 2 layers of QAOA parameters (4 parameters total)
np.random.seed(24)
gamma_angles = np.array([0.1, 0.5], requires_grad=True)
beta_angles = np.array([0.3, 0.2], requires_grad=True)

# Define the expected energy loss function
def energy_loss_fn(g_angles, b_angles):
    probs = qaoa_annealer(g_angles, b_angles)

    # Calculate the expectation value of the material Hamiltonian energy
    # Energy = Sum (Probability_of_State * Energy_of_State)
    state_energies = qml.matrix(H_cost).diagonal().real
    expected_energy = np.sum(probs * state_energies)
    return expected_energy

optimizer = qml.AdamOptimizer(stepsize=0.08)

print("Optimizing Quantum Annealing Schedule...")
print("------------------------------------------")

for step in range(31):
    # Simultaneously optimize the time parameters for the cost and mixer fields
    (gamma_angles, beta_angles), current_energy = optimizer.step_and_cost(
        energy_loss_fn, gamma_angles, beta_angles
    )

    if step % 10 == 0:
        print(f"Step {step:2d} | Target Material System Energy: {current_energy:.5f} eV")

print("------------------------------------------")

### Step 4: Extracting the Stable Material Structure

In [ ]:
final_probabilities = qaoa_annealer(gamma_angles, beta_angles)
best_state_index = np.argmax(final_probabilities)
best_state_binary = format(best_state_index, f'0{num_grains}b')

# Map 0 -> Up spin (+1), 1 -> Down spin (-1)
crystal_profile = ["↑" if char == '0' else "↓" for char in best_state_binary]

print("\n--- Material Discovery Verification ---")
print(f"Optimal Crystal Domain Layout Vector: {crystal_profile}")
print(f"Confidence of Quantum Convergence:   {final_probabilities[best_state_index]*100:.2f}%")

## High-Impact Research Starting Points
If you want to use this pipeline to target peer-reviewed materials science journals, here are three viable directions:

**High-Entropy Alloy (HEA) Phase Prediction**
High-entropy alloys are complex metals made by mixing five or more elements in roughly equal amounts. Predicting whether an HEA will form a clean, single-phase solid solution or split into brittle, fractured phases is an open problem.
*   Research Project: Modify the grid structure to map a multi-element mixing graph. Instead of binary spins ($Z_i = \pm 1$), your students can scale the code to a Quantum Potts Model using multi-qubit encodings per node to represent multiple elements ($A, B, C, D$). This allows them to search for phase stability layouts.

**Compare QAOA vs. Classical Simulated Annealing (SA)**
A major point of debate in literature is whether quantum tunneling actually beats classical thermal hopping in complex material spaces.
*   Research Project: Write a standard Python script for Classical Simulated Annealing using a Metropolis-Hastings Monte Carlo loop. Create an intentionally rugged energy landscape filled with sharp local energy wells. Benchmark the number of steps your classical script takes to find the ground state against the QAOA simulation depth ($p$-layers).

**Deploying to Hardware via Analog Amazon Braket Providers**
While universal gate-based QPUs run QAOA, you can access native Analog Quantum Annealing on AWS using QuEra's Aquila neutral-atom device.